# Lot 46 — G5 Deep-Target Training & Strategic Qualification

Notebook Colab **orchestrateur mince**. La logique scientifique réside dans `packages/songo_ai/training/lot46.py` et `apps/trainer/scripts/run_srn_lot46.py`.

Ce notebook ne contourne aucun gate : si les résultats finaux Lot45, les checksums, les checkpoints G4R ou CUDA manquent, l'entraînement reste interdit. Il ne promeut jamais G5 automatiquement.

In [ ]:
# Configuration unique du run — modifier uniquement ces chemins.
from pathlib import Path

REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_BRANCH = 'dev'
REPO = Path('/content/songo-fish')
DRIVE_ROOT = Path('/content/drive/MyDrive/songo-ai')
LOT45_INPUTS = DRIVE_ROOT / 'inputs/lot45_inputs.tar.gz'
LOT46_INPUTS = DRIVE_ROOT / 'inputs/lot46_inputs.tar.gz'
LOT45_ARCHIVE = DRIVE_ROOT / 'exports/lot45_results.tar.gz'
LOT46_OUTPUT = DRIVE_ROOT / 'experiments/lot46_g5_training'
LOT46_EXPORT = DRIVE_ROOT / 'exports/lot46_results.tar.gz'
SEED = 20264601
print({'repo': str(REPO), 'lot45_inputs': str(LOT45_INPUTS), 'lot46_inputs': str(LOT46_INPUTS), 'lot45_archive': str(LOT45_ARCHIVE), 'output': str(LOT46_OUTPUT)})

In [ ]:
# Colab seulement : monter Drive.
from google.colab import drive
drive.mount('/content/drive')

## 1. Dépôt et environnement

Le dépôt doit déjà être cloné dans `REPO`. Aucun code scientifique n'est défini dans le notebook.

In [ ]:
import os, subprocess, sys
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
    # /content est une copie de travail jetable : réaligner les fichiers suivis sur la branche distante.
    # Les artefacts d'entraînement non suivis (checkpoints/datasets) sont conservés.
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
runner = REPO / 'apps/trainer/scripts/run_srn_lot46.py'
assert runner.is_file(), f'Runner Lot46 absent sur {REPO_BRANCH} @ {commit[:10]}: {runner}'
os.chdir(REPO)
# Préserver la build CUDA fournie par Colab : ne pas laisser pip réinstaller torch depuis l'index CPU.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest>=8.0'], check=True)
import torch
nvidia = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
environment = {'commit': commit, 'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'nvidia_smi': nvidia.returncode == 0, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(environment)
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU CUDA absent. Dans Colab: Exécution > Modifier le type d’exécution > Accélérateur matériel > T4 GPU, '
        'puis redémarrer la session et relancer depuis la première cellule. '
        f'État détecté: {environment}'
    )

## 2. Import transactionnel des résultats Lot45

L'archive est vérifiée avant extraction. L'archive d'entrée `lot45_inputs.tar.gz` n'est pas acceptée comme résultat.

In [ ]:
import hashlib, json, shutil, tarfile, tempfile

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def verify_archive(path):
    assert path.is_file(), f'Archive absente: {path}'
    sidecar = Path(str(path) + '.sha256')
    assert sidecar.is_file(), f'Checksum absent: {sidecar}'
    actual = file_sha256(path)
    expected = sidecar.read_text().split()[0]
    assert actual == expected, f'Checksum invalide pour {path.name}: {actual} != {expected}'
    return actual

def safe_extract(archive_path, destination):
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, f'Chemin dangereux: {member.name}'
        archive.extractall(destination, filter='data')

inputs_sha = verify_archive(LOT45_INPUTS)
safe_extract(LOT45_INPUTS, REPO)
input_manifest = REPO / 'lot45_input_manifest.json'
if input_manifest.is_file():
    manifest = json.loads(input_manifest.read_text())
    bad = [name for name, digest in manifest.get('files', {}).items() if not (REPO / name).is_file() or file_sha256(REPO / name) != digest]
    assert not bad, f'Entrées Lot45 corrompues ou absentes: {bad[:10]}'

lot46_inputs_sha = verify_archive(LOT46_INPUTS)
safe_extract(LOT46_INPUTS, REPO)
lot46_manifest_path = REPO / 'lot46_input_manifest.json'
assert lot46_manifest_path.is_file(), 'lot46_input_manifest.json absent du bundle Lot46'
lot46_manifest = json.loads(lot46_manifest_path.read_text())
bad = [name for name, digest in lot46_manifest['files'].items() if not (REPO / name).is_file() or file_sha256(REPO / name) != digest]
assert not bad, f'Entrées Lot46 corrompues ou absentes: {bad}'

actual = verify_archive(LOT45_ARCHIVE)

destination = REPO / 'data/experiments/lot45_g5_target_generation'
with tempfile.TemporaryDirectory(dir='/content') as temporary:
    staging = Path(temporary)
    safe_extract(LOT45_ARCHIVE, staging)
    matches = list(staging.rglob('dataset/g5_deep_autonomous_reanalysis_v1.jsonl.gz'))
    assert len(matches) == 1, f'Dataset Lot45 final introuvable ou ambigu: {matches}'
    extracted_root = matches[0].parent.parent
    assert (extracted_root / 'decision.json').is_file() and (extracted_root / 'dataset_manifest.json').is_file()
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(extracted_root, destination)
print({'lot45_inputs_sha256': inputs_sha, 'lot46_inputs_sha256': lot46_inputs_sha, 'lot45_results_sha256': actual, 'extracted_to': str(destination)})

## 3. Tests puis audit/preflight

Le runner écrit tous les rapports dans un répertoire propre à l'expérience.

In [ ]:
# Idempotent : une cellule relancée seule restaure d'abord les quatre checkpoints G4R.
lot46_inputs_sha = verify_archive(LOT46_INPUTS)
safe_extract(LOT46_INPUTS, REPO)
required_g4 = [
    REPO / 'data/experiments/lot34r_g4_retry/checkpoints/control/step-08000.pt',
    REPO / 'data/experiments/lot34r_g4_retry/checkpoints/control/step-12000.pt',
    REPO / 'data/experiments/lot34r_g4_retry/checkpoints/pool/step-06000.pt',
    REPO / 'data/experiments/lot34r_g4_retry/checkpoints/pool/step-12000.pt',
]
required_lot46a_data = [REPO / 'data/d_rl/lot11_g1_selected_seed_20260924.jsonl']
missing_g4 = [str(path) for path in required_g4 if not path.is_file()]
assert not missing_g4, f'Bundle Lot46 extrait mais checkpoints absents: {missing_g4}'
missing_lot46a_data = [str(path) for path in required_lot46a_data if not path.is_file()]
assert not missing_lot46a_data, f'Bundle Lot46 extrait mais datasets de test absents: {missing_lot46a_data}'
manifest = json.loads((REPO / 'lot46_input_manifest.json').read_text())
bad_g4 = [name for name, digest in manifest['files'].items() if not (REPO / name).is_file() or file_sha256(REPO / name) != digest]
assert not bad_g4, f'Fichiers Lot46 invalides après extraction: {bad_g4}'
print({'lot46_inputs_sha256': lot46_inputs_sha, 'g4_checkpoints_ready': len(required_g4), 'lot46a_test_datasets_ready': len(required_lot46a_data)})

env = {**os.environ, 'PYTHONPATH': f'{REPO / "packages"}:{REPO / "apps/trainer/scripts"}'}
def run_checked(command):
    result = subprocess.run(command, env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'Commande échouée ({result.returncode}): {command}')
    return result

run_checked([sys.executable, '-m', 'pytest', '-q', 'packages/songo_ai/tests/test_lot46_g5_training.py', 'packages/songo_ai/tests/test_lot46a_training.py'])
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'prepare', '--output', str(LOT46_OUTPUT)])
preflight = json.loads((LOT46_OUTPUT / 'preflight_report.json').read_text())
display(preflight)
assert preflight['SCIENTIFIC_TRAINING_ALLOWED'] == 'YES', preflight

## 4. Smoke end-to-end

Forward, losses Policy/Value masquées, backward, clipping, optimizer step et checkpoint atomique réel.

In [ ]:
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'smoke', '--output', str(LOT46_OUTPUT)])
smoke = json.loads((LOT46_OUTPUT / 'smoke_test_report.json').read_text())
display(smoke)
assert smoke['status'] == 'PASS', smoke

## 5. Lot46A — configuration et préparation

Choisir une configuration miniature. Les quatre familles sont disponibles dans `configs/lot46a/`. Aucun entraînement scientifique long ni aucune arène n'est lancé par le notebook.

In [ ]:
import json

EXPERIMENT_CONFIG = REPO / 'configs/lot46a/deep_policy_smoke.json'
CONFIRM_FULL_TRAINING = 'NO'  # doit rester NO pendant Lot46A
RUN_MINI_TRAINING = False     # passer manuellement à True pour le run de 40 étapes
assert EXPERIMENT_CONFIG.is_file(), EXPERIMENT_CONFIG
configuration = json.loads(EXPERIMENT_CONFIG.read_text())
assert not configuration.get('confirm_full_training'), 'Lot46A interdit un entraînement scientifique complet'
display(configuration)
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'prepare-a', '--config', str(EXPERIMENT_CONFIG)])

## 6. Micro-overfit et reprise déterministe

Ces tests sont courts et font partie de la qualification d'ingénierie Lot46A.

In [ ]:
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'micro-overfit', '--config', str(EXPERIMENT_CONFIG)])
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'resume-test', '--config', str(EXPERIMENT_CONFIG)])

## 7. Mini-entraînement optionnel, reprise et validation

`RUN_MINI_TRAINING` autorise uniquement la configuration miniature préenregistrée. `CONFIRM_FULL_TRAINING` reste obligatoirement à `NO` pour Lot46A.

In [ ]:
if RUN_MINI_TRAINING:
    assert CONFIRM_FULL_TRAINING == 'NO'
    assert configuration['max_steps'] <= 100 and not configuration['confirm_full_training']
    run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'train', '--config', str(EXPERIMENT_CONFIG)])
    run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'validate', '--config', str(EXPERIMENT_CONFIG)])
    run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'status', '--config', str(EXPERIMENT_CONFIG)])
else:
    print('Mini-entraînement non lancé. Mettre RUN_MINI_TRAINING=True puis relancer cette cellule.')

## 8. Inspection et export technique Lot46A

L'export est nommé `lot46a_engineering_results.tar.gz`; il ne constitue pas un résultat scientifique Lot46.

In [ ]:
run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'inspect-checkpoint', '--config', str(EXPERIMENT_CONFIG)])
if RUN_MINI_TRAINING:
    run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'finalize-a', '--config', str(EXPERIMENT_CONFIG)])
    run_checked([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'export-a', '--config', str(EXPERIMENT_CONFIG), '--bundle', str(DRIVE_ROOT / 'exports/lot46a_engineering_results.tar.gz')])
else:
    print('Finalisation différée : exécuter d’abord le mini-entraînement Lot46A.')
print('CONTROLLED STOP — Lot46B non lancé; aucune arène; aucune promotion G5.')